In [ ]:
"""
Py-Microgrid Hybrid System Simulation Example
-----------------------------------
This example demonstrates how to:
1. Set up a hybrid system simulation with 5-component architecture
2. Download solar and wind resource data
3. Configure system parameters
4. Run optimization (single location or multiple locations)
5. Analyze and save results

5-Component Architecture:
- PV: Solar photovoltaic panels
- Wind: Wind turbines
- Battery: Energy storage system
- Genset: Backup generator (renamed from 'grid')
- Grid: True electrical grid connection (new)

Required files:
- Base YAML configuration file
- CSV file containing location data
"""

import os
import pandas as pd
from typing import Dict, List, Any

from py_microgrid.utilities import ConfigManager
from py_microgrid.utilities.keys import set_developer_nrel_gov_key
from py_microgrid.tools.optimization import SystemOptimizer, LoadAnalyzer
from py_microgrid.tools.analysis.bos import EconomicCalculator
from py_microgrid.simulation.resource_files import ResourceDataManager

class HybridOptimizer:
    """Wrapper class for hybrid system optimization with 5-component architecture."""
    
    def __init__(self, 
                 yaml_file_path: str,
                 api_key: str,
                 email: str,
                 project_lifetime: int = 25,
                 discount_rate: float = 0.0588,
                 enable_flexible_load: bool = False,  # Default to no flexible load
                 max_load_reduction_percentage: float = 0.2):
        """
        Initialize hybrid system optimizer.
        
        Args:
            yaml_file_path: Path to YAML configuration file
            api_key: NREL API key for resource data
            email: Email for API authentication
            project_lifetime: Project lifetime in years
            discount_rate: Discount rate for economic calculations
            enable_flexible_load: Whether to enable flexible load management
            max_load_reduction_percentage: Maximum load reduction (if flexible load enabled)
        """
        self.yaml_file_path = yaml_file_path
        self.api_key = api_key
        self.email = email
        
        # Set up components
        self.resource_manager = ResourceDataManager(api_key, email)
        self.economic_calculator = EconomicCalculator(discount_rate, project_lifetime)
        self.system_optimizer = SystemOptimizer(
            yaml_file_path, 
            self.economic_calculator,
            enable_flexible_load=enable_flexible_load,
            max_load_reduction_percentage=max_load_reduction_percentage
        )
        
        # Define optimization bounds for 5-component architecture
        self.bounds = [
            (5000, 50000),    # PV capacity (kW)
            (1, 50),          # Wind turbines (1MW each)
            (5000, 30000),    # Battery energy capacity (kWh)
            (1000, 10000),    # Battery power capacity (kW)
            (17000, 30000),   # Genset capacity (kW) - backup generator
            (5000, 20000),    # Grid interconnect (kW) - grid connection
        ]

    def process_location(self, latitude: float, longitude: float, location_id: str = "") -> Dict[str, Any]:
        """Process single location optimization with 5-component architecture."""
        print(f"\nProcessing location {location_id} at ({latitude}, {longitude})")
        
        try:
            # Download solar and wind resource data
            solar_path = self.resource_manager.download_solar_data(
                latitude, longitude, "2020"
            )
            wind_path = self.resource_manager.download_wind_data(
                latitude, longitude, "20200101", "20201231"
            )
            
            # Update configuration with location and resource data
            config = self.system_optimizer.config_manager.load_yaml_safely(self.yaml_file_path)
            config['site']['data']['lat'] = latitude
            config['site']['data']['lon'] = longitude
            config['site']['solar_resource_file'] = solar_path.replace('\\', '/')
            config['site']['wind_resource_file'] = wind_path.replace('\\', '/')
            self.system_optimizer.config_manager.save_yaml_safely(config, self.yaml_file_path)
            
            # Set initial conditions (10% of range) for 5 components
            initial_conditions = [
                [bound[0] + (bound[1] - bound[0]) * 0.1 for bound in self.bounds]
            ]
            
            # Run optimization
            best_result = self.system_optimizer.optimize_system(self.bounds, initial_conditions)
            
            if best_result:
                # Print and return results
                print("\nBest configuration found:")
                print(f"  PV Capacity: {best_result.get('PV Capacity (kW)', 0):.0f} kW")
                print(f"  Wind Capacity: {best_result.get('Wind Turbine Capacity (kW)', 0):.0f} kW")
                print(f"  Battery Energy: {best_result.get('Battery Energy Capacity (kWh)', 0):.0f} kWh")
                print(f"  Battery Power: {best_result.get('Battery Power Capacity (kW)', 0):.0f} kW")
                print(f"  Genset Capacity: {best_result.get('Genset Capacity (kW)', 0):.0f} kW (backup generator)")
                print(f"  Grid Interconnect: {best_result.get('Grid Interconnect (kW)', 0):.0f} kW (grid connection)")
                print(f"  LCOE: ${best_result.get('System LCOE ($/kWh)', 0):.4f}/kWh")
                print(f"  System Cost: ${best_result.get('System Cost ($)', 0):,.0f}")
                print(f"  Demand Met: {best_result.get('Demand Met Percentage', 0):.1f}%")
                
                return {
                    'Latitude': latitude,
                    'Longitude': longitude,
                    'Location ID': location_id,
                    **best_result
                }
            else:
                print("Optimization failed to converge")
                return {}
                
        except Exception as e:
            print(f"Error processing location: {str(e)}")
            return {}

def main():
    """Main execution."""
    # ======= Configuration =======
    # File paths
    yaml_file_path = "../input_yaml/input_file_chunk_0.yaml"  # Base configuration file
    csv_path = "../deposit_data/auCopper_chunk_0.csv"         # Location data
    output_path = "../simulation_results/simulation_results_chunk_0.csv"  # Results output
    
    # API credentials (replace with your own)
    api_key = "YOUR-NREL-API-KEY"  # Get from https://developer.nrel.gov/
    email = "your.email@example.com"  # Required for NERL API
    
    # Set NREL API key
    set_developer_nrel_gov_key(api_key)
    
    # ======= Initialize Optimizer =======
    optimizer = HybridOptimizer(
        yaml_file_path=yaml_file_path,
        api_key=api_key,
        email=email,
        enable_flexible_load=False,      # Set to True to enable flexible load
        max_load_reduction_percentage=0.2  # Only used if flexible load is enabled
    )
    
    # ======= Run Single Location Example =======
    # Example coordinates (replace with your location)
    test_location = {
        'latitude': -33.5265,
        'longitude': 149.1588,
        'id': "TEST_LOCATION"
    }
    
    result = optimizer.process_location(
        latitude=test_location['latitude'],
        longitude=test_location['longitude'],
        location_id=test_location['id']
    )
    
    if result:
        print("\nSingle location optimization successful!")
        print(f"LCOE: ${result['System LCOE ($/kWh)']:.4f}/kWh")
        print(f"Total System Cost: ${result['System Cost ($)']:,.2f}")
        print(f"Demand Met: {result['Demand Met Percentage']:.1f}%")
        
        # Display 5-component breakdown
        print("\n5-Component System Breakdown:")
        print(f"  PV: {result.get('PV Capacity (kW)', 0):.0f} kW")
        print(f"  Wind: {result.get('Wind Turbine Capacity (kW)', 0):.0f} kW")
        print(f"  Battery: {result.get('Battery Energy Capacity (kWh)', 0):.0f} kWh / {result.get('Battery Power Capacity (kW)', 0):.0f} kW")
        print(f"  Genset: {result.get('Genset Capacity (kW)', 0):.0f} kW (backup generator)")
        print(f"  Grid: {result.get('Grid Interconnect (kW)', 0):.0f} kW (grid connection)")
    
    # ======= Run Multiple Locations Example =======
    # Comment out if you only want to run single location
    """
    # Load location data
    locations = pd.read_csv(csv_path)
    
    # Process all locations
    results = []
    for _, row in locations.iterrows():
        result = optimizer.process_location(
            latitude=row['DEPOSIT_LATITUDE'],
            longitude=row['DEPOSIT_LONGITUDE'],
            location_id=row['DEPOSIT_UID']
        )
        if result:
            results.append(result)
    
    # Save and summarize results
    if results:
        results_df = pd.DataFrame(results)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        results_df.to_csv(output_path, index=False)
        
        print("\nOptimization complete. Summary:")
        print(f"Total locations processed: {len(results)}")
        print(f"Average LCOE: ${results_df['System LCOE ($/kWh)'].mean():.4f}/kWh")
        print(f"Best LCOE: ${results_df['System LCOE ($/kWh)'].min():.4f}/kWh")
        
        # Display component statistics
        print("\nComponent Statistics:")
        print(f"  Average PV: {results_df['PV Capacity (kW)'].mean():.0f} kW")
        print(f"  Average Wind: {results_df['Wind Turbine Capacity (kW)'].mean():.0f} kW")
        print(f"  Average Battery: {results_df['Battery Energy Capacity (kWh)'].mean():.0f} kWh")
        print(f"  Average Genset: {results_df['Genset Capacity (kW)'].mean():.0f} kW")
        print(f"  Average Grid: {results_df['Grid Interconnect (kW)'].mean():.0f} kW")
    """

if __name__ == "__main__":
    main()